<a href="https://colab.research.google.com/github/VinayaSharada/KateelLearningDemosToStudents/blob/main/TreasuryAnalytics/ARAgingCollectionsPrioritizer/ar_aging_collections_prioritizer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AR Aging & Collections Prioritizer

This notebook is a transparent, cell-by-cell walkthrough of the same logic behind the **AR Aging & Collections Prioritizer** Claude Skill demoed in Session 3, *"Receivables Intelligence: From Aging Reports to Predictive Collections"*. Instead of a chat response, you get to see the aging buckets, the weighted risk score, and the four-segment framework computed directly in front of you.

Nothing here is "AI magic" — it is deterministic, inspectable Python: aging math, a weighted score, and a handful of rules. The point of the notebook is to make that visible, and to let you compare its output side by side with the "Receivables Strategy by Customer Type" framework from the slide.

## Learning goals

- See that AR aging, DSO estimation, and collections prioritization can be fully explained as arithmetic and rules — no hidden model.
- Compare the notebook's four-segment output against the "Receivables Strategy by Customer Type" slide framework.
- Re-run the same scoring engine with different weights and observe how the collections worklist re-ranks.
- Understand where this approach's scale limits are (see the optional section at the end).

## 1. Setup

Opening this notebook via the "Open in Colab" badge only loads the notebook file itself — it does not clone the repo, so the two files this notebook depends on (`prioritize_collections.py`, `sample_ar_aging.csv`) will not be present yet. The cell below fetches them from GitHub if they are missing, and does nothing if you already have them locally (e.g. you cloned the repo).

In [ ]:
import os
import urllib.request

RAW_BASE_URL = "https://raw.githubusercontent.com/VinayaSharada/KateelLearningDemosToStudents/main/TreasuryAnalytics/ARAgingCollectionsPrioritizer/"

for _fname in ["prioritize_collections.py", "sample_ar_aging.csv"]:
    if not os.path.exists(_fname):
        urllib.request.urlretrieve(RAW_BASE_URL + _fname, _fname)
        print(f"Downloaded {_fname}")
    else:
        print(f"Found {_fname} locally")

## Step-by-Step Explanation

### What this cell is doing
1. Imports the Python libraries that support data handling, modeling, or visualization in the next steps.
2. Exports an artifact so the result can be shared, reviewed, or used in a later workflow step.
3. Shows an immediate checkpoint so students can verify that the previous transformation worked as expected.
4. Key code cues in this cell include `import os`, which sets the direction for the rest of the cell.

### How to interpret the result
- Use the imported library list to explain which tools are responsible for tables, charts, and model behavior later in the notebook.
- Check whether the exported file captures the right level of evidence for classroom discussion or follow-up analysis.
- Ask students what business decision would change if this output moved materially up, down, or in an unexpected direction.


In [ ]:
import sys
sys.path.append(".")

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from datetime import date

from prioritize_collections import load_rows, compute, parse_weights, write_priority_csv, write_summary_csv

# Consistent segment colors used across every chart in this notebook
SEGMENT_COLORS = {
    "Distressed / high-risk": "#dc2626",
    "Strategic & high-value": "#2563eb",
    "Habitually late but solvent": "#d97706",
    "Low-risk repetitive": "#16a34a",
}
SEGMENT_ORDER = list(SEGMENT_COLORS.keys())


def dollar_axis(ax, axis="y"):
    formatter = mticker.FuncFormatter(lambda x, _pos: f"${x:,.0f}")
    (ax.yaxis if axis == "y" else ax.xaxis).set_major_formatter(formatter)


## Step-by-Step Explanation

### What this cell is doing
1. Imports the Python libraries that support data handling, modeling, or visualization in the next steps.
2. Defines reusable logic so the notebook can repeat the same analysis consistently across scenarios.
3. Key code cues in this cell include `import sys`, which sets the direction for the rest of the cell.

### How to interpret the result
- Use the imported library list to explain which tools are responsible for tables, charts, and model behavior later in the notebook.
- A function definition does not usually produce business output yet; the important question is what inputs it expects and what output it will later generate.
- Ask students what business decision would change if this output moved materially up, down, or in an unexpected direction.


## 2. Choose sample data or bring your own AR export

This notebook now supports a self-service scoring flow similar to the invoice-level collections notebook. By default it uses the classroom-safe sample file. If you want to score your own sanitized AR export, set `USE_SAMPLE_DATA = False` in the next cell and upload a CSV with the same core columns.

The notebook also writes `ar_aging_data_template.csv` so you can download a starter file, map your own export to it, and upload that version back into Colab.

In [ ]:
try:
    from google.colab import files as colab_files
    IN_COLAB = True
except ImportError:
    colab_files = None
    IN_COLAB = False

USE_SAMPLE_DATA = True
TEMPLATE_PATH = "ar_aging_data_template.csv"
TEMPLATE_COLUMNS = [
    "customer_id", "customer_name", "invoice_id", "invoice_date", "due_date",
    "amount_usd", "dispute_flag", "avg_days_late_last_4_quarters",
    "annual_revenue_contribution_usd", "credit_rating_note",
]

template_df = pd.read_csv("sample_ar_aging.csv")[TEMPLATE_COLUMNS].head(8).copy()
template_df.to_csv(TEMPLATE_PATH, index=False)
print(f"Wrote {TEMPLATE_PATH} for optional BYOD scoring.")
if IN_COLAB:
    print("Running in Colab. You can download the template and optionally upload your own AR export.")
else:
    print(f"Running outside Colab — find the template at: {TEMPLATE_PATH}")

INPUT_FILE = "sample_ar_aging.csv"
if not USE_SAMPLE_DATA:
    print("Upload a sanitized AR CSV for scoring. Required columns: invoice_id, invoice_date, due_date, amount_usd, and one of customer_id/customer_name.")
    if IN_COLAB:
        uploaded = colab_files.upload()
        INPUT_FILE = next(iter(uploaded)) if uploaded else TEMPLATE_PATH
    else:
        INPUT_FILE = TEMPLATE_PATH
        print("Not in Colab, so the notebook will look for your mapped file locally. Update INPUT_FILE if needed.")

print(f"Selected input file: {INPUT_FILE}")


## Step-by-Step Explanation

### What this cell is doing
1. Detects whether the notebook is running in Colab so upload and download behavior can stay self-service friendly.
2. Writes a starter template that users can adapt for their own AR export.
3. Lets the user choose between the classroom-safe sample file and their own sanitized upload.

### How to interpret the result
- If `USE_SAMPLE_DATA = True`, you stay on the guided sample path.
- If `USE_SAMPLE_DATA = False`, the notebook becomes a scoring tool for your own mapped export.
- The selected input file is the file the rest of the notebook will score, so this is the most important self-service checkpoint in the notebook.


## 3. Load selected data

In [ ]:
rows = load_rows(INPUT_FILE)
raw_df = pd.DataFrame(rows)
print(f"Loaded {len(rows)} rows from {INPUT_FILE}")
print("Columns:", ", ".join(raw_df.columns))
raw_df.head()


## Step-by-Step Explanation

### What this cell is doing
1. Loads whichever file you selected in the previous step: the sample file or your own mapped AR export.
2. Prints the column list so you can confirm the schema before scoring.
3. Shows the first few rows as a self-service checkpoint before any scoring logic runs.

### How to interpret the result
- Confirm that the column names match your expectations before moving on.
- If you uploaded your own file, this is where you catch mapping mistakes early.
- The notebook scores what it sees here, so any data-quality issue at this step will flow into the prioritization output.


## 4. Run the calculation

`compute()` is the same function the Claude Skill calls. It parses each invoice, computes days past due and the aging bucket, scores risk with the default weights, and assigns one of the four segments.

In [ ]:
AS_OF = date(2026, 6, 30)
weights = parse_weights(None)  # defaults: days_past_due=0.40, amount=0.25, dispute=0.15, history=0.15, value_protect=0.05

valid, errors = compute(rows, as_of=AS_OF, weights=weights)
print(f"{len(valid)} invoices scored, {len(errors)} skipped due to parse errors")

df = pd.DataFrame(valid)
df[["_customer_key", "invoice_id", "_amount", "_days_past_due", "_aging_bucket", "_segment", "_risk_score", "_priority_rank"]].head(10)

## Step-by-Step Explanation

### What this cell is doing
1. Loads or generates the dataset that the rest of the analysis depends on.
2. Shows an immediate checkpoint so students can verify that the previous transformation worked as expected.
3. Key code cues in this cell include `AS_OF = date(2026, 6, 30)`, which sets the direction for the rest of the cell.

### How to interpret the result
- Review the rows and columns carefully because this dataset defines what the model or analysis is allowed to learn from.
- Use this checkpoint to confirm that the structure, sample values, and labels still make business sense.
- Ask students what business decision would change if this output moved materially up, down, or in an unexpected direction.


## 5. Aging & DSO summary

These are the same portfolio-level numbers the Skill writes to `aging_summary.csv`: total open AR, a DSO estimate (amount-weighted days outstanding), and the disputed total.

The bar chart below breaks AR by aging bucket, and — to make cross-chart comparison easier — stacks each bucket by segment, using the same segment colors used later in the segmentation and worklist charts.

In [ ]:
total_ar = df["_amount"].sum()
total_disputed = df.loc[df["_dispute"], "_amount"].sum()
dso_estimate = (df["_amount"] * df["_days_outstanding"].clip(lower=0)).sum() / total_ar

print(f"Total open AR:      ${total_ar:,.0f}")
print(f"DSO estimate:       {dso_estimate:.1f} days")
print(f"Total disputed:     ${total_disputed:,.0f} ({100 * total_disputed / total_ar:.1f}% of AR)")

## Step-by-Step Explanation

### What this cell is doing
1. Shows an immediate checkpoint so students can verify that the previous transformation worked as expected.
2. Key code cues in this cell include `total_ar = df["_amount"].sum()`, which sets the direction for the rest of the cell.

### How to interpret the result
- Use this checkpoint to confirm that the structure, sample values, and labels still make business sense.
- Ask students what business decision would change if this output moved materially up, down, or in an unexpected direction.


In [ ]:
bucket_order = ["current", "1-30", "31-60", "61-90", "90+"]
pivot = (
    df.pivot_table(index="_aging_bucket", columns="_segment", values="_amount", aggfunc="sum", fill_value=0)
    .reindex(bucket_order)
    .reindex(columns=SEGMENT_ORDER, fill_value=0)
)

fig, ax = plt.subplots(figsize=(8, 5))
bottom = pd.Series(0.0, index=pivot.index)
for segment in SEGMENT_ORDER:
    ax.bar(pivot.index, pivot[segment], bottom=bottom, label=segment, color=SEGMENT_COLORS[segment])
    bottom += pivot[segment]

dollar_axis(ax)
ax.set_title("AR by Aging Bucket (stacked by segment)")
ax.set_xlabel("Aging bucket")
ax.set_ylabel("Amount")
ax.legend(loc="upper right", fontsize=8)
plt.tight_layout()
plt.show()

## Step-by-Step Explanation

### What this cell is doing
1. Aggregates, profiles, or reshapes the data so important operating patterns become visible.
2. Creates a chart so learners can inspect structure, trend, dispersion, or risk visually.
3. Key code cues in this cell include `bucket_order = ["current", "1-30", "31-60", "61-90", "90+"]`, which sets the direction for the rest of the cell.

### How to interpret the result
- Interpret the summary output as evidence about concentration, distribution, and unusual behavior before jumping to decisions.
- The right interpretation is usually about relative shape, outliers, and direction of movement rather than memorizing exact pixel-level detail.
- Ask students what business decision would change if this output moved materially up, down, or in an unexpected direction.


## 6. Segmentation

Compare this notebook's segment totals against the **"Receivables Strategy by Customer Type"** framework from the Session 3 slide. The rules are documented in `reference/segmentation_rules.md` in the Skill folder — reproduced here as a table so you can check the notebook's output against it directly.

| Segment | Rule (evaluated in this order) | Best response | Treasury objective |
|---|---|---|---|
| **Distressed / high-risk** | `days_past_due >= 60` AND (disputed OR `avg_days_late_last_4_quarters >= 45`), OR any negative credit note | Advance payment, collateral, or reduced exposure | Protect downside, reduce bad-debt risk |
| **Strategic & high-value** | Not Distressed, AND top-quartile `annual_revenue_contribution_usd` | Commercial escalation and coordinated resolution | Protect the relationship without losing cash discipline |
| **Habitually late but solvent** | Not Distressed or Strategic, AND `avg_days_late_last_4_quarters >= 20` OR (`days_past_due >= 30` with no dispute) | Tighter terms, structured follow-up, selective credit limits | Improve reliability of inflows |
| **Low-risk repetitive** | Everything else | Automation and workflow reminders | Lower collections cost |

In [ ]:
action_by_segment = {
    "Distressed / high-risk": "Advance payment, collateral, or reduced exposure",
    "Strategic & high-value": "Commercial escalation and coordinated resolution",
    "Habitually late but solvent": "Tighter terms, structured follow-up, selective credit limits",
    "Low-risk repetitive": "Automation and workflow reminders",
}

segment_summary = (
    df.groupby("_segment")
    .agg(customer_invoice_count=("invoice_id", "count"), amount_usd=("_amount", "sum"))
    .reindex(SEGMENT_ORDER)
)
segment_summary["pct_of_total"] = (100 * segment_summary["amount_usd"] / total_ar).round(1)
segment_summary["recommended_action"] = [action_by_segment[s] for s in segment_summary.index]
segment_summary

## Step-by-Step Explanation

### What this cell is doing
1. Aggregates, profiles, or reshapes the data so important operating patterns become visible.
2. Key code cues in this cell include `action_by_segment = {`, which sets the direction for the rest of the cell.

### How to interpret the result
- Interpret the summary output as evidence about concentration, distribution, and unusual behavior before jumping to decisions.
- Ask students what business decision would change if this output moved materially up, down, or in an unexpected direction.


In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
sizes = segment_summary["amount_usd"]
colors = [SEGMENT_COLORS[s] for s in sizes.index]
wedges, _ = ax.pie(sizes, colors=colors, startangle=90, wedgeprops=dict(width=0.4))
ax.legend(wedges, [f"{s}  (${v:,.0f})" for s, v in sizes.items()], loc="center left", bbox_to_anchor=(1, 0.5), fontsize=8)
ax.set_title("AR $ by Segment")
plt.tight_layout()
plt.show()

## Step-by-Step Explanation

### What this cell is doing
1. Creates a chart so learners can inspect structure, trend, dispersion, or risk visually.
2. Key code cues in this cell include `fig, ax = plt.subplots(figsize=(6, 6))`, which sets the direction for the rest of the cell.

### How to interpret the result
- The right interpretation is usually about relative shape, outliers, and direction of movement rather than memorizing exact pixel-level detail.
- Ask students what business decision would change if this output moved materially up, down, or in an unexpected direction.


## 7. Collections worklist

The worklist ranks invoices by **risk-weighted $ exposure** (`risk_score * amount`) — the same ranking the Skill writes to `collections_priority.csv`. This is what a collections analyst would actually work from Monday morning.

In [ ]:
TOP_N_WORKLIST = 15

df["_exposure"] = df["_risk_score"] * df["_amount"]
worklist = df.sort_values("_priority_rank").head(TOP_N_WORKLIST).copy()
worklist["recommended_action"] = worklist["_segment"].map(action_by_segment)

display_cols = ["_priority_rank", "_customer_key", "invoice_id", "_amount", "_days_past_due", "_aging_bucket", "_segment", "_risk_score", "recommended_action"]
worklist[display_cols]

## Step-by-Step Explanation

### What this cell is doing
1. Runs a focused analysis step that transforms the current notebook state into the next working result.
2. Key code cues in this cell include `TOP_N_WORKLIST = 15`, which sets the direction for the rest of the cell.

### How to interpret the result
- Interpret the output by asking what changed from the previous step and why that matters for the decision the notebook supports.
- Ask students what business decision would change if this output moved materially up, down, or in an unexpected direction.


In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))
plot_df = worklist.iloc[::-1]  # highest priority at the top of the chart
labels = plot_df["_customer_key"] + " — " + plot_df["invoice_id"]
colors = [SEGMENT_COLORS[s] for s in plot_df["_segment"]]
ax.barh(labels, plot_df["_exposure"], color=colors)

dollar_axis(ax, axis="x")
ax.set_title(f"Top {TOP_N_WORKLIST} Accounts by Risk-Weighted $ Exposure")
ax.set_xlabel("risk_score × amount")
plt.tight_layout()
plt.show()

## Step-by-Step Explanation

### What this cell is doing
1. Creates a chart so learners can inspect structure, trend, dispersion, or risk visually.
2. Key code cues in this cell include `fig, ax = plt.subplots(figsize=(9, 6))`, which sets the direction for the rest of the cell.

### How to interpret the result
- The right interpretation is usually about relative shape, outliers, and direction of movement rather than memorizing exact pixel-level detail.
- Ask students what business decision would change if this output moved materially up, down, or in an unexpected direction.


## 8. CFO scenario

This is the number used in the CFO-translation paragraph in the chat demo: if collections focused only on the top N accounts by priority, how much of total AR would that cover?

In [ ]:
TOP_N_CFO = 10  # change this and re-run to see the CFO number move

top_accounts = df.sort_values("_priority_rank").head(TOP_N_CFO)
top_amount = top_accounts["_amount"].sum()
top_pct = 100 * top_amount / total_ar

print(f"Top {TOP_N_CFO} accounts by priority represent ${top_amount:,.0f}, or {top_pct:.1f}% of total open AR (${total_ar:,.0f}).")

## Step-by-Step Explanation

### What this cell is doing
1. Shows an immediate checkpoint so students can verify that the previous transformation worked as expected.
2. Key code cues in this cell include `TOP_N_CFO = 10  # change this and re-run to see the CFO number move`, which sets the direction for the rest of the cell.

### How to interpret the result
- Use this checkpoint to confirm that the structure, sample values, and labels still make business sense.
- Ask students what business decision would change if this output moved materially up, down, or in an unexpected direction.


## 9. Try your own weights

The default weights (`days_past_due=0.40, amount=0.25, dispute=0.15, history=0.15, value_protect=0.05`) are one policy choice, not the only one. Try a cash-crunch scenario that weights `amount` more heavily, and see how the worklist reorders.

In [ ]:
cash_crunch_weights = parse_weights("days_past_due=0.20,amount=0.50,dispute=0.15,history=0.10,value_protect=0.05")

valid_reweighted, _ = compute(rows, as_of=AS_OF, weights=cash_crunch_weights)
df_reweighted = pd.DataFrame(valid_reweighted)

before = df.sort_values("_priority_rank").head(10)["invoice_id"].tolist()
after = df_reweighted.sort_values("_priority_rank").head(10)["invoice_id"].tolist()

comparison = pd.DataFrame({"default_weights_rank": range(1, 11), "invoice_id (default)": before,
                            "cash_crunch_rank": range(1, 11), "invoice_id (cash-crunch)": after})
comparison

## Step-by-Step Explanation

### What this cell is doing
1. Loads or generates the dataset that the rest of the analysis depends on.
2. Key code cues in this cell include `cash_crunch_weights = parse_weights("days_past_due=0.20,amount=0.50,dispute=0.15,history=0`, which sets the direction for the rest of the cell.

### How to interpret the result
- Review the rows and columns carefully because this dataset defines what the model or analysis is allowed to learn from.
- Ask students what business decision would change if this output moved materially up, down, or in an unexpected direction.


**Discussion:** Which accounts moved into or out of the top 10 when `amount` was weighted more heavily than `days_past_due`? Does that make sense given what each invoice looks like — or does it surface a case where a "bigger $" bias overrides a genuinely more urgent smaller account?

## 10. Reflection

1. Where would this notebook's output fit into your organization's 100-day collections improvement program?
2. If you had to justify the top-10 worklist to management, which numbers from this notebook would you lead with, and why?
3. Which data column is your organization currently missing that this notebook assumes you already have (dispute flag, average days late, revenue contribution, credit rating note)?

---

## 10. Optional: scale-testing (skip if short on time)

This section is intentionally separated from the walkthrough above — it does not change any of the results shown so far. It generates a larger *synthetic* AR file with the same column schema as `sample_ar_aging.csv` and times how long `compute()` takes to run on it.

**Read the caveat before drawing conclusions:** the current `prioritize_collections.py` is plain Python (no vectorization). Timing it at 50K–100K rows tells you whether it is "still fine for a mid-size company's AR book" — it is **not** evidence that this approach scales to millions of rows or real-time, multi-entity treasury data. That would require a pandas/polars rewrite, which is out of scope for this teaching notebook. Only report what you actually measure in the cell below.

## 10. Export a reusable collections queue

This step turns the notebook output into files that can be reused in downstream treasury automation or governance exercises.


In [ ]:
PRIORITY_EXPORT_PATH = "collections_action_queue.csv"
SUMMARY_EXPORT_PATH = "ar_aging_summary.csv"

write_priority_csv(valid, PRIORITY_EXPORT_PATH)
write_summary_csv(valid, AS_OF, SUMMARY_EXPORT_PATH)

segment_owner_map = {
    "Distressed / high-risk": "Special Collections Desk",
    "Strategic & high-value": "Relationship Manager + Treasury",
    "Habitually late but solvent": "Collections Team",
    "Low-risk repetitive": "Automated Reminder Queue",
}

export_queue = worklist.copy()
export_queue["customer_id"] = export_queue.get("customer_id", export_queue["_customer_key"])
export_queue["customer_name"] = export_queue.get("customer_name", export_queue["_customer_key"])
export_queue["invoice_amount"] = export_queue["_amount"].round(2)
export_queue["due_date"] = pd.to_datetime(export_queue["_due_date"]).dt.strftime("%Y-%m-%d")
export_queue["expected_payment_date"] = pd.to_datetime(export_queue["_due_date"]) + pd.to_timedelta(export_queue["_days_past_due"].clip(lower=0), unit="D")
export_queue["expected_payment_date"] = export_queue["expected_payment_date"].dt.strftime("%Y-%m-%d")
export_queue["predicted_days_vs_due"] = export_queue["_days_past_due"].clip(lower=0)
export_queue["late_risk_probability"] = (export_queue["_risk_score"] / 100).round(4)
export_queue["priority_band"] = pd.cut(
    export_queue["_risk_score"],
    bins=[-0.1, 35, 60, 80, 100],
    labels=["Routine", "Monitor closely", "Escalate this week", "Immediate action"],
)
export_queue["recommended_action"] = export_queue["recommended_action"]
export_queue["assigned_owner"] = export_queue["_segment"].map(segment_owner_map)
export_queue["escalation_date"] = (pd.Timestamp(AS_OF) + pd.to_timedelta(2, unit="D")).strftime("%Y-%m-%d")
export_queue["approval_required"] = (export_queue["_segment"] == "Strategic & high-value") | (export_queue["_risk_score"] >= 80)

pack_cols = [
    "invoice_id", "customer_id", "customer_name", "invoice_amount", "due_date",
    "expected_payment_date", "predicted_days_vs_due", "late_risk_probability",
    "priority_band", "recommended_action", "assigned_owner", "escalation_date", "approval_required",
]
export_queue[pack_cols].to_csv(PRIORITY_EXPORT_PATH, index=False)

print(f"Wrote {PRIORITY_EXPORT_PATH} and {SUMMARY_EXPORT_PATH}")
export_queue[pack_cols].head(10)


## Step-by-Step Explanation

### What this cell is doing
1. Exports an artifact so the result can be shared, reviewed, or used in a later workflow step.
2. Shows an immediate checkpoint so students can verify that the previous transformation worked as expected.
3. Key code cues in this cell include `PRIORITY_EXPORT_PATH = "collections_action_queue.csv"`, which sets the direction for the rest of the cell.

### How to interpret the result
- Check whether the exported file captures the right level of evidence for classroom discussion or follow-up analysis.
- Use this checkpoint to confirm that the structure, sample values, and labels still make business sense.
- Ask students what business decision would change if this output moved materially up, down, or in an unexpected direction.


In [ ]:
import csv
import time
import numpy as np

def generate_synthetic_ar(path, n_rows, n_customers=None, seed=42):
    """Write a synthetic AR file with the same column schema as sample_ar_aging.csv."""
    rng = np.random.default_rng(seed)
    n_customers = n_customers or max(1, n_rows // 4)

    customer_ids = [f"SYN{cid:06d}" for cid in range(n_customers)]
    customer_revenue = {cid: float(rng.lognormal(mean=13.0, sigma=1.0)) for cid in customer_ids}
    customer_avg_late = {cid: float(np.clip(rng.normal(20, 15), 0, 75)) for cid in customer_ids}

    terms_choices = [15, 30, 45, 60]
    anchor = date(2026, 6, 30)

    with open(path, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow([
            "customer_id", "customer_name", "invoice_id", "invoice_date", "due_date",
            "amount_usd", "dispute_flag", "avg_days_late_last_4_quarters",
            "annual_revenue_contribution_usd", "credit_rating_note",
        ])
        for i in range(n_rows):
            customer_id = customer_ids[rng.integers(0, n_customers)]
            invoice_offset_days = int(rng.integers(5, 150))
            invoice_date = anchor - pd.Timedelta(days=invoice_offset_days)
            terms = int(rng.choice(terms_choices))
            due_date = invoice_date + pd.Timedelta(days=terms)
            amount = round(float(rng.lognormal(mean=9.5, sigma=1.1)), 2)
            dispute = "Y" if rng.random() < 0.08 else "N"
            credit_note = "Downgraded - watchlist" if rng.random() < 0.02 else ""

            writer.writerow([
                customer_id, f"Synthetic Customer {customer_id}", f"SYNINV-{i:07d}",
                invoice_date.isoformat(), due_date.isoformat(), amount,
                dispute, round(customer_avg_late[customer_id], 1),
                round(customer_revenue[customer_id], 2), credit_note,
            ])

SCALE_TEST_ROWS = 50_000  # try 100_000 as well
SCALE_TEST_PATH = "synthetic_ar_scale_test.csv"

generate_synthetic_ar(SCALE_TEST_PATH, SCALE_TEST_ROWS)
scale_rows = load_rows(SCALE_TEST_PATH)
print(f"Generated {len(scale_rows):,} synthetic rows at {SCALE_TEST_PATH}")

## Step-by-Step Explanation

### What this cell is doing
1. Imports the Python libraries that support data handling, modeling, or visualization in the next steps.
2. Loads or generates the dataset that the rest of the analysis depends on.
3. Defines reusable logic so the notebook can repeat the same analysis consistently across scenarios.
4. Key code cues in this cell include `import csv`, which sets the direction for the rest of the cell.

### How to interpret the result
- Use the imported library list to explain which tools are responsible for tables, charts, and model behavior later in the notebook.
- Review the rows and columns carefully because this dataset defines what the model or analysis is allowed to learn from.
- Ask students what business decision would change if this output moved materially up, down, or in an unexpected direction.


In [ ]:
start = time.perf_counter()
scale_valid, scale_errors = compute(scale_rows, as_of=AS_OF, weights=parse_weights(None))
elapsed = time.perf_counter() - start

print(f"compute() on {len(scale_rows):,} rows took {elapsed:.2f}s ({len(scale_rows) / elapsed:,.0f} rows/sec)")
print(f"Valid: {len(scale_valid):,}  Skipped: {len(scale_errors):,}")

## Step-by-Step Explanation

### What this cell is doing
1. Shows an immediate checkpoint so students can verify that the previous transformation worked as expected.
2. Key code cues in this cell include `start = time.perf_counter()`, which sets the direction for the rest of the cell.

### How to interpret the result
- Use this checkpoint to confirm that the structure, sample values, and labels still make business sense.
- Ask students what business decision would change if this output moved materially up, down, or in an unexpected direction.


**What this does and does not show:** the timing above is specific to this run, this machine, and `SCALE_TEST_ROWS`. It is evidence about *this* teaching notebook at *this* size — nothing more. Re-run with `SCALE_TEST_ROWS = 100_000` if you want a second data point, but do not extrapolate a "scales to millions of rows" claim from it.